# Train a diffusion model with Dew

This notebook trains an image diffusion model with the library's public pieces: a dataset, a `Process` built from a preset, a DiT model, `DiffusionObjective`, and `Trainer`. It then samples from the trained weights, restores the checkpoint, and samples again.

There are two routes through the same code:

- **Offline route (default, runs here).** A generated stripe dataset at 16 pixels, a tiny DiT, and a short run. It exercises every transition (train, evaluate, sample, checkpoint, restore) on a CPU in a few minutes. The images are not interesting; the workflow is.
- **Oxford Flowers route.** Set `ROUTE = "flowers"` after preparing the dataset once as ArrayRecords (see the data section). Training 64-pixel flowers needs a GPU or TPU and roughly an hour; that route was not executed while writing this notebook.

In [ ]:
# On Colab: install dew and the JAX build for the runtime. Locally this cell is a no-op.
try:
    import google.colab  # noqa: F401
    import subprocess, sys
    try:
        import jax
        tpu = any("tpu" in str(d).lower() for d in jax.devices())
    except Exception:
        tpu = False
    extra = "jax[tpu]" if tpu else "jax[cuda12]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "dew-ml[tfds] @ git+https://github.com/AshishKumar4/dew", extra])
except ImportError:
    pass

In [ ]:
ROUTE = "offline"          # "offline" runs anywhere; "flowers" needs the prepared dataset and an accelerator
RUN_DIR = "runs/02-diffusion"

if ROUTE == "offline":
    IMAGE_SIZE = 16
    BATCH_SIZE = 8
    STEPS = 30
    MODEL = dict(patch_size=4, emb_features=32, num_layers=2, num_heads=2, mlp_ratio=2)
    DTYPE = "float32"
    ATTENTION = "xla"
    SAMPLE_STEPS = 8
else:
    FLOWERS_PATH = "~/dew-data/tfds-arrayrecord/oxford_flowers102/2.1.1"  # builder.data_dir from preparation
    IMAGE_SIZE = 64
    BATCH_SIZE = 64
    STEPS = 20_000
    MODEL = dict(patch_size=4, emb_features=384, num_layers=8, num_heads=6)
    DTYPE = "bfloat16"
    ATTENTION = "auto"
    SAMPLE_STEPS = 40
SAMPLES = 8
SEED = 0

In [ ]:
import jax

print(jax.devices())
print(jax.default_backend())

## The data

A `Dataset` is two iterator factories plus a record count: `train()` opens an endless shuffled stream and `val()` opens one finite pass. Batches carry uint8 images in `[0, 255]`; the objective converts to `[-1, 1]` itself. A training stream that reports its position through `get_state`/`set_state` is what lets a checkpoint resume the data where it stopped; Dew's built-in loaders do this through Grain, and the offline stream below does it with a batch counter.

The offline route builds its images in memory: horizontal or vertical stripes in a random colour, one label per orientation. The flowers route reads prepared TFDS ArrayRecords. Preparation happens once, in a separate environment that has TensorFlow (the reader in the training environment does not need it):

```bash
uv venv --python 3.13 .venv-tfds-prepare
uv pip install --python .venv-tfds-prepare/bin/python tensorflow-datasets==4.9.10 tensorflow==2.21.0 scipy
TFDS_DATA_DIR=~/dew-data/tfds-arrayrecord .venv-tfds-prepare/bin/python -c "
import tensorflow_datasets as tfds
b = tfds.builder('oxford_flowers102'); b.download_and_prepare(file_format='array_record'); print(b.data_dir)"
```

`FLOWERS_PATH` is the directory that command prints.

In [ ]:
import itertools

import numpy as np
from dew.data import Dataset, Loading


def stripe_images(count, size, seed):
    """uint8 [count, size, size, 3] stripe images and a 0/1 orientation label."""
    rng = np.random.default_rng(seed)
    images = np.zeros((count, size, size, 3), np.uint8)
    labels = rng.integers(0, 2, count)
    for i in range(count):
        colour = rng.integers(64, 256, 3)
        period = int(rng.integers(2, 5))
        rows = (np.arange(size) // period) % 2 == 0
        # label 0: bands across the image vary with the row index; label 1: with the column
        pattern = rows[:, None] if labels[i] == 0 else rows[None, :]
        images[i][np.broadcast_to(pattern, (size, size))] = colour
    return images, labels.astype(np.int32)


if ROUTE == "offline":
    train_images, train_labels = stripe_images(256, IMAGE_SIZE, seed=SEED)
    val_images, val_labels = stripe_images(BATCH_SIZE, IMAGE_SIZE, seed=SEED + 1)

    class StripeBatches:
        """An endless stream of random batches that can report and restore
        its position, which is what a checkpoint needs to resume the data."""

        def __init__(self):
            self.index = 0

        def __iter__(self):
            return self

        def __next__(self):
            rows = np.random.default_rng(SEED + self.index).choice(
                len(train_images), BATCH_SIZE, replace=False)
            self.index += 1
            return {"image": train_images[rows], "label": train_labels[rows]}

        def get_state(self):
            return str(self.index).encode()

        def set_state(self, state):
            self.index = int(state.decode())

    data = Dataset(train=StripeBatches,
                   val=lambda: iter([{"image": val_images, "label": val_labels}]),
                   records=len(train_images), batch=BATCH_SIZE)
else:
    from dew.data import OxfordFlowers
    data = OxfordFlowers(path=FLOWERS_PATH, image_size=IMAGE_SIZE, val_batches=2,
                         loading=Loading(workers=4, threads=4, read_buffer=16, worker_buffer=2)
                         ).load(batch=BATCH_SIZE)

print(f"{data.records} training records, {data.steps_per_epoch} steps per epoch")

In [ ]:
from PIL import Image

try:
    from IPython.display import display
except ModuleNotFoundError:  # running the cells as a plain script
    def display(image):
        print(f"image {image.width}x{image.height}")


def show_grid(frames, cols=8, scale=4):
    """Tile uint8 [N, H, W, 3] frames into one image and display it inline."""
    frames = np.asarray(frames)
    rows = (len(frames) + cols - 1) // cols
    padded = np.zeros((rows * cols, *frames.shape[1:]), np.uint8)
    padded[:len(frames)] = frames
    h, w, c = frames.shape[1:]
    grid = padded.reshape(rows, cols, h, w, c).transpose(0, 2, 1, 3, 4).reshape(rows * h, cols * w, c)
    image = Image.fromarray(grid)
    display(image.resize((image.width * scale, image.height * scale), Image.NEAREST))
    return image


batch = next(iter(data.val()))
print(batch["image"].shape, batch["image"].dtype)
show_grid(batch["image"][:8])

## The process and the model

A diffusion model is three choices: the noise levels it trains on, the noise levels it walks when sampling, and what the network predicts. `presets.EDM()` is a configuration value; calling it builds the `Process` that binds a schedule, a prediction transform, and a loss weighting together. EDM (Karras et al., 2022) draws training noise levels from a log-normal distribution and samples on the Karras sigma spacing.

`models.build("simple_dit", ...)` constructs the DiT from the registry. `dtype` is the compute dtype and `attention_impl` picks the attention kernel; the parameter tree is the same either way. `InputSpec(Field("image", (H, W, 3)))` names the batch field the model generates and its per-example shape, with no conditions: this model learns only what the images look like.

In [ ]:
from dew import Field, InputSpec, models, presets

process = presets.EDM()()
model = models.build("simple_dit", **MODEL, output_channels=3, dtype=DTYPE, attention_impl=ATTENTION)
inputs = InputSpec(Field("image", (IMAGE_SIZE, IMAGE_SIZE, 3)))
print(type(process.schedule).__name__, type(process.prediction).__name__)

## The objective and the trainer

`DiffusionObjective` owns the diffusion computation: sample a noise level, corrupt the image, run the model, weight the error. Its `sampler`, `guidance` and `steps` configure the preview it generates at evaluation time, not the number of optimizer updates.

`Trainer` owns the rest: it differentiates the loss, applies the Optax optimizer, keeps the EMA copy the objective asks for (`ema_decay`), and writes checkpoints. `fit` trains to a total step count. `eval_every` runs the validation pass, in which this objective generates one image per validation record from the EMA weights; a metric consumes those images. `psnr` compares each generated image with the validation image at the same position, which for unconditional samples is a rough number rather than a quality score, but it shows the evaluation path running. A tracker (`WandbTracker`) would also receive a preview grid; without one, nothing is drawn during training.

In [ ]:
import optax
from dew import Checkpoints, Trainer, metrics
from dew.objectives.diffusion import DiffusionObjective
from dew.sampling import Euler, EulerAncestral

objective = DiffusionObjective(model, process, inputs, ema_decay=0.99,
                               sampler=EulerAncestral(), guidance=None, steps=SAMPLE_STEPS)
trainer = Trainer(objective, optax.adamw(2e-4), key=jax.random.key(SEED),
                  checkpoints=Checkpoints(RUN_DIR))

variables = jax.eval_shape(objective.init, jax.random.key(0))
n_params = sum(int(np.prod(x.shape)) for x in jax.tree_util.tree_leaves(variables["params"]))
print(f"{n_params / 1e6:.2f}M parameters")

In [ ]:
state = trainer.fit(data, steps=STEPS, log_every=max(1, STEPS // 5),
                    eval_every=STEPS, checkpoint_every=STEPS, metrics=(metrics.psnr(),))
print("attempted steps:", int(state.step), "| optimizer updates:", int(state.updates))

## Sample from the trained weights

Sampling reverses the corruption: start from Gaussian noise at the top noise level and walk it down with a solver. `process.denoiser` wraps the model and its parameters into the function a solver reads, `process.noise` draws the starting point with the right variance, and `sample` runs the solver over `steps` points in one compiled scan. `state.averaged` is the variables tree with the EMA weights in place of the live ones.

In [ ]:
from dew.sampling import sample


def save_grid(images, path):
    """Images in [-1, 1] as a saved and displayed uint8 grid."""
    frames = np.clip((np.asarray(images) + 1) * 127.5, 0, 255).astype(np.uint8)
    show_grid(frames).save(path)
    return path


def generate(params, count, key, solver=EulerAncestral()):
    denoise = process.denoiser(model, objective.trainable(params), {})
    x_T = process.noise(key, (count, IMAGE_SIZE, IMAGE_SIZE, 3))
    return sample(denoise, x_T, SAMPLE_STEPS, solver=solver, key=key)


images = generate(state.averaged, SAMPLES, jax.random.key(1))
print(images.shape, float(images.min()), float(images.max()))
save_grid(images, f"{RUN_DIR}/samples.png")

## Restore the checkpoint

A checkpoint holds the state, not the code: rebuild the same objective and trainer, and `place()` restores the latest step onto the devices. The restored EMA weights produce the same images as the ones sampled above when given the same key, which is the check that the round trip preserved them.

In [ ]:
restored_trainer = Trainer(objective, optax.adamw(2e-4), key=jax.random.key(SEED),
                           checkpoints=Checkpoints(RUN_DIR))
restored, _, _ = restored_trainer.place()
print("restored step:", int(restored.step))

restored_images = generate(restored.averaged, SAMPLES, jax.random.key(1))
print("max difference from the live samples:",
      float(np.max(np.abs(np.asarray(restored_images) - np.asarray(images)))))
save_grid(restored_images, f"{RUN_DIR}/samples-restored.png")

## Where to go next

`recipes/diffusion/train.py` runs this workflow from the command line with `run.json` written beside the checkpoints, which is what `TextToImage.from_run` reads to rebuild a trained model for sampling. [Notebook 03](03-text-to-image-with-guidance.ipynb) adds a text condition and classifier-free guidance; [notebook 04](04-samplers-and-schedules.ipynb) compares the solvers on this notebook's checkpoint.